In [5]:
import HeST as hest
import HeST.Amherst_split_cpd as examp
import numpy as np
import matplotlib.pyplot as plt
import HeST.Detection as detection
%matplotlib qt

import astropy.stats as astat
from scipy.interpolate import interp1d

ModuleNotFoundError: No module named 'HeST_Core'

In [2]:

detector = examp.Amherst_split_cpd 


QP_conditions= detector.get_surface_conditions()
nCPDs = detector.get_nCPDs()
for i in range(nCPDs):
    QP_conditions.append( (detector.get_CPD(i)).get_surface_condition() )

QP_conditions.append( detector.liquid_surface )
detector.set_QP_reflection_prob(0.98)
detector.set_diffuse_prob(0.0)

In [3]:
def plot_stacked_hist(evap):
    cpd_1_ints = np.unique(evap.bounce_flag[0])

    cpd_2_ints = np.unique(evap.bounce_flag[1])
    masks_cpd_1 = np.empty((len(cpd_1_ints), len(evap.bounce_flag[0])), dtype = int)
    masks_cpd_2 = np.empty((len(cpd_2_ints), len(evap.bounce_flag[1])), dtype = int)
    cpd1_times = evap.arrivalTimes_us[0]
    cpd2_times = evap.arrivalTimes_us[1]
    plt.figure()
    for ii, bounce_num in enumerate(cpd_1_ints):
        mask = evap.bounce_flag[0] == bounce_num 
        plt.hist(cpd1_times[mask],bins = 200, range = [0,3000], alpha= 0.9, label = 'cpd1, bounce = ' + str(bounce_num - 1))
    plt.title('Arrival times, with bounce')
    plt.xlabel('Time [us]')
    plt.legend()
    plt.show()
    plt.figure()
    for ii, bounce_num in enumerate(cpd_2_ints):
        mask = evap.bounce_flag[1] == bounce_num
        plt.hist(cpd2_times[mask], bins = 200, range = [0.0, 3000.0],alpha = 0.9, label= 'cpd2, bounce = ' + str(bounce_num - 1) )
    plt.title('Arrival Times, with bounce')
    plt.xlabel('Time [us]')
    plt.legend()
    plt.show()

In [4]:
pos = [-0.1, 0.0, 1.5]
useMap = False
dir_unnormalized = np.array([0, 1, 4])
dir =dir_unnormalized/np.sum(dir_unnormalized**2)
evap = hest.GetEvaporationSignal( detector, 1, *pos, useMap=useMap, debug=True, debug_dir=dir,  plot_3d=True, choose_momentum=True, momentum_choice= 1.3)
plt.figure()
plt.hist(evap.arrivalTimes_us[0], bins=200, range=[0, 3000], histtype='step', color='r', lw=2, label = 'CPD1')
plt.hist(evap.arrivalTimes_us[1], bins=200, range=[0, 3000], histtype='step', color='b', lw=2, label = 'CPD2')
plt.legend()
plt.title('Arrival Times')
plt.xlabel('time [us]')
plt.ylabel('Counts/bin')
plt.show()


Starting at [-0.1, 0.0, 1.5]
~~~~~~~~~~~~~~~~ Finding Surface Intersection ~~~~~~~~~~~~~~~~~


[[-0.1]
 [ 0. ]
 [ 1.5]]
[[0.        ]
 [0.05882353]
 [0.23529412]]
XY [[ True  True  True  True  True  True  True  True  True  True  True  True
   True  True  True  True  True  True  True  True  True  True  True  True
   True  True  True  True  True  True  True  True  True  True  True  True
   True  True  True  True  True  True  True  True  True  True  True  True
   True  True  True  True  True  True  True  True  True  True  True  True
   True  True  True  True  True  True  True  True  True  True  True  True
   True  True  True  True  True  True  True  True  True  True  True  True
   True  True  True  True  True  True  True  True  True  True  True  True
   True  True  True  True  True  True  True  True  True  True  True  True
   True  True  True  True  True  True  True  True  True  True  True  True
   True  True  True  True  True  True  True  True  True  True  True  True
   True  True  True 

In [21]:
@np.vectorize
def critical_angle(Energy, momentum, binding_energy = 0.00062e-3):
    m =  3.725472e6 #He mass in keV/c^2
    c = 2.998e8
    print(np.sqrt(2 * m * Energy - binding_energy)/momentum)
    return np.arcsin(-1 * np.sqrt(2 * m * (Energy - binding_energy))momentum)
def GetInterpFunc(d_path):
    """Creates an linear interpolation function from data found at the file path below,, giving us the ability to convert from resistance to temperature. 
    returns:
        Interpoltion function: If input exceeds range of the data function returns a NaN"""
    data = np.loadtxt(d_path, delimiter=',')
    X = data[:,0]
    Y = data[:,1]
    return interp1d(X,Y, kind = 'linear')

def QP_dispersion(p ):
    """generate the energy of the particle in meV

    Args:
        p (int): the momentum of the quasiparticle of interest 
        interp (funct): function to relate momentum and energy  
    """
    interp = GetInterpFunc('./dispersion_data.csv')
    energy = interp(p)
    return energy * 1e-3 #This is in eV 


momentum = np.linspace(0.145, 4.7, 100)
energy = QP_dispersion(momentum) *1e-3 
plt.plot(momentum, critical_angle(energy, momentum, binding_energy=0.00062e-3))
plt.xlim(0, 4.7)

5.352862620222836
5.352862620222836
4.985293299241731
4.614694926433562
4.31164563358465
4.056376467684137
3.8490102310223397
3.658853270308426
3.5017550175221017
3.3617081067200405
3.233002499546087
3.122045872610331
3.01993981254747
2.9250270690947575
2.8386175193725744
2.75898488296788
2.684251013387396
2.610899640273196
2.543187860558035
2.47990559507221
2.418377863240865
2.360592561824813
2.309223786529602
2.2557557239208057
2.203080811192667
2.15177663655989
2.102938296227964
2.054409686517805
2.0078676519310164
1.9625976959188807
1.9185812030859974
1.8757207127042417
1.8339563018184424
1.793684959478592
1.7542174772395758
1.7161986708343053
1.6791567834707677
1.6432298026701968
1.6084443170519598
1.5745279847310008
1.5419448452304745
1.5101116726401946
1.479140181907689
1.4492308626295227
1.4200655069453398
1.3914201311899224
1.363975309331583
1.3363683641631832
1.3093674653940128
1.2836698630944234
1.2578050311843147
1.232362766203395
1.2072687567095381
1.1823623405870964
1.157

(0.0, 4.7)

In [19]:
plt.hist(evap.num_bounces, bins=20 )
plt.yscale('log')
plt.xlabel('number of bounces')
plt.ylabel('Counts per num of bounce')
plt.title('Understanding if we are gettign the expected number of bounces')
print(np.mean(evap.num_bounces))

48.309


In [ ]:
pos = [0., 0., 1.5]
useMap = False
evap = hest.GetEvaporationSignal( detector, 10, *pos, useMap=useMap, debug=False, plot_3d=True)


In [ ]:
plot_stacked_hist(evap)


In [ ]:
pos = [0., 0., 1.5]
useMap = False
evap = hest.GetEvaporationSignal( detector,4*pos, useMap=useMap, debug=False, plot_3d=True)
plot_stacked_hist(evap=evap)


In [ ]:
# need to prep two example arrays 
p = np.array([0,0,0])
he = np.array([1,2,3])
p = np.append(p, he)



In [ ]:
n = np.array([[0,1,2],[4,5,6]])
print(n[:,0])

In [ ]:
probs = np.linspace(0, .9, 10)
counts = np.empty_like(probs)
for ii, p in enumerate(probs):   
    pos = [0., 0., 1.5]
    detector.set_QP_reflection_prob(p)
    
    useMap = False
    evap = hest.GetEvaporationSignal( detector, 50000,*pos, useMap=useMap)
    counts[ii] = len(evap.arrivalTimes_us[0])

plt.plot(probs, counts)
plt.title('Logical Check. As the number of reflections goes up, so the should the number of events we have')
plt.legend()



In [ ]:
import HeST.Detection as detection
import os
import numpy as np
#The detector geometry is defined from the point of view of particle paths.
# We essentially want to define various "surface conditions" where the particle paths are obstructed
# These functions also carry a "boundary_type", so that we can keep track if the particle is obstructed by
# a CPD, or a wall, and how it may reflect off of a given wall.

def sensor1_conditions(x, y, z):
    boundary_type = "CPD0"
    radius = 3.8
    height = 3.3
    return (x*x + y*y < radius*radius) & (x>0)& (z < height)| (x*x + y*y >= radius*radius) , boundary_type

def sensor2_conditions(x, y, z):
    boundary_type = "CPD1"
    radius = 3.8
    height = 3.3
    return (x*x + y*y < radius*radius) &  (x<0)&(z < height)| (x*x + y*y >= radius*radius) , boundary_type





baseline_noise = [0., 0.]
phonon_conversion = 0.25
cpd1 = detection.VCPD(sensor1_conditions, baseline_noise, phonon_conversion)
cpd2 = detection.VCPD(sensor2_conditions, baseline_noise, phonon_conversion)





def wall_conditions(x, y, z):
    boundary_type = "XY"
    radius = 3. #cm
    height = 2.75 #cm
    return ((x*x + y*y < radius*radius) & (z < height) ) | (z > height), boundary_type

def bottom_conditions(x, y, z):
    boundary_type = "Z"
    bottom = 0. #cm
    return (z > bottom), boundary_type

def liquid_surface(x, y, z):
    boundary_type = "Liquid"
    height = 2.75 #cm
    return (z < height), boundary_type

def liquid_conditions(x, y, z):
    height = 2.75 #cm
    radius = 3. #cm
    bottom = 0. #cm
    return ((x*x + y*y < radius*radius) & (z < height) & (z > bottom))
   

Amherst_split_cpd = detection.VDetector([wall_conditions, bottom_conditions], liquid_surface=liquid_surface, liquid_conditions=liquid_conditions, CPDs=[cpd1, cpd2], adsorption_gain=6.0e-3, evaporation_eff=0.60)

 

In [ ]:
def sensor2_conditions(x, y, z):
    boundary_type = "CPD1"
    radius = 3.8
    height = 3.3
    return (x*x + y*y < radius*radius) &  (x<0)&(z < height)| (x*x + y*y >= radius*radius) , boundary_type

detector= detection.VDetector([wall_conditions, bottom_conditions], liquid_surface=liquid_surface, liquid_conditions=liquid_conditions, CPDs=[cpd1, cpd2], adsorption_gain=6.0e-3, evaporation_eff=0.60)


cpd_2_x_bounds = np.linspace(0,-3.1, 10)
signals_cpd1 = np.empty_like(cpd_2_x_bounds)
signals_cpd2 = np.empty_like(cpd_2_x_bounds)


for ii, bound in enumerate(cpd_2_x_bounds):
    pos = [0., 0., 1.5]
    def sensor2_conditions(x, y, z):
        boundary_type = "CPD1"
        radius = 3.8
        height = 3.3
        print(bound)
        return (x*x + y*y < radius*radius) &  (x<bound)&(z < height)| (x*x + y*y >= radius*radius) , boundary_type
    cpd1 = detection.VCPD(sensor1_conditions, baseline_noise, phonon_conversion)
    cpd2 = detection.VCPD(sensor2_conditions, baseline_noise, phonon_conversion)


    detector = detection.VDetector([wall_conditions, bottom_conditions], liquid_surface=liquid_surface, liquid_conditions=liquid_conditions, CPDs=[cpd1, cpd2], adsorption_gain=6.0e-3, evaporation_eff=0.60)

    useMap = False
    evap = hest.GetEvaporationSignal( detector, 40000, *pos, useMap=useMap)
    print( evap.area_eV, evap.chArea_eV, evap.coincidence, len(evap.arrivalTimes_us))
    signals_cpd1[ii] = len(evap.arrivalTimes_us[0])
    signals_cpd2[ii] = len(evap.arrivalTimes_us[1])

plt.plot(cpd_2_x_bounds, signals_cpd1, label = 'Without changing size')
plt.plot(cpd_2_x_bounds, signals_cpd2, label = 'CPD2, with changing size')
plt.legend()
plt.xlabel('Where cpd1 starts')
plt.ylabel('Number of hits per')
plt.title('Testing if cpd 1 and 2 separation is working')


In [ ]:
def GetInterpFunc():
    """Creates an linear interpolation function from data found at the file path below,, giving us the ability to convert from resistance to temperature. 
    returns:
        Interpoltion function: If input exceeds range of the data function returns a NaN"""
    data = np.loadtxt('./dispersion_data.csv', delimiter=',')
    X = data[:,0]
    Y = data[:,1]
    return interp1d(X,Y, kind = 'linear')

In [ ]:
interp = GetInterpFunc()

In [ ]:
plt.subplot(211)
data = np.loadtxt('./dispersion_data.csv', delimiter=',')
v_data = np.loadtxt('./velocity_data.csv', delimiter=',')
X = data[:,0]
Y = data[:,1]
plt.plot(X, Y, label = 'Dispersion Curve')
plt.legend()
plt.show()
plt.subplot(212)
x = v_data[:,0]
y = v_data[:,1]
plt.plot(x[:-3], y[:-3], label = 'velocity')
plt.legend()
plt.show()

In [ ]:
def k_squared_distribution(u):
    N = (4.7**3 - .15**3)/3
    c = .15**3/(3 * N)
    return (3 * N* (u + c))**(1/3)

u = np.random.uniform(size=10000)
dist = k_squared_distribution(u)
plt.hist(dist, bins = 200)
plt.title('Distribution of K-values')
plt.xlabel('momenta')
plt.ylabel('counts/bin')